<a href="https://colab.research.google.com/github/hiten4/Week8/blob/main/Week8_wednesday.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W8 Wednesday Assignment (Using Twitter_Data.csv)
CNN + Embeddings

In [1]:

!pip install sentence-transformers -q


In [2]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity


In [3]:

# Load your dataset
df = pd.read_csv('/content/Twitter_Data.csv')

df = df.rename(columns={'clean_text':'text', 'category':'hate_speech'})
df['text'] = df['text'].astype(str)

print(df.head())
print(df['hate_speech'].value_counts())


                                                text  hate_speech
0  when modi promised “minimum government maximum...         -1.0
1  talk all the nonsense and continue all the dra...          0.0
2  what did just say vote for modi  welcome bjp t...          1.0
3  asking his supporters prefix chowkidar their n...          1.0
4  answer who among these the most powerful world...          1.0
hate_speech
 1.0    72250
 0.0    55213
-1.0    35510
Name: count, dtype: int64


In [4]:

# Handle missing values
df['text'] = df['text'].fillna("")
df['hate_speech'] = df['hate_speech'].fillna(0)


In [5]:

# TF-IDF + Model
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['text'])
y = df['hate_speech']

clf = LogisticRegression(max_iter=200)
clf.fit(X, y)


LogisticRegression(max_iter=200)

In [6]:

# Embeddings
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['text'].tolist())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:

def semantic_search(query):
    q_emb = model.encode([query])
    scores = cosine_similarity(q_emb, embeddings)[0]
    top_idx = np.argsort(scores)[-5:]
    return df.iloc[top_idx][['text','hate_speech']]

semantic_search("hate speech example")


,text,hate_speech
119994,says speech did not violate the,0.0
28274,hate speeches call people anti nationals when ...,-1.0
128389,win only for his hatred speech,1.0
520,deserves punishment for his hate speeches,-1.0
134080,modi speech were filled with hates,1.0


In [8]:

def moderation_pipeline(text):
    vec = tfidf.transform([text])
    pred = clf.predict(vec)[0]

    if pred == 1:
        return "Flagged"
    else:
        return semantic_search(text)

moderation_pipeline("bad comment")


,text,hate_speech
39534,\npathetic,-1.0
80568,funny,1.0
102904,karma bitch,0.0
88906,retarded,-1.0
1241,joke,0.0
